<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.8-deep-research-agents/notebooks/exercises-9_8.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.8: Deep Research Agents

*Module 9 · AI Agents & Autonomous Systems*

> A search engine finds links. A deep research agent understands a topic : generates sub-questions, searches multiple sources, evaluates credibility, extracts key findings, identifies contradictions, and synthesizes everything into a structured report — with citations. This is the architecture behind Gemini Deep Research, Perplexity, and Claude’s research mode.

`Sub-Questions` · `Multi-Source` · `Source Eval` · `Synthesis` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.8.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Question Decomposition — Break Complex Questions Apart — `01_decompose.py`
2. Step 2 — Multi-Source Search + Source Evaluation — `02_search_evaluate.py`
3. Step 3 — Full Deep Research Agent — End to End — `03_deep_research.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Question Decomposition — Break Complex Questions Apart

The LLM splits a broad question into focused sub-questions for targeted search.

**`01_decompose.py`** · _Block 1 of 3_

QUESTION DECOMPOSITION — Break into sub-questions


In [ ]:
# QUESTION DECOMPOSITION — Break into sub-questions
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

def decompose(question, n=4):
    """Break a research question into focused sub-questions."""
    resp = model.generate_content(
        f"Break this research question into {n} focused sub-questions.\n"
        f"Each should target a different aspect.\n"
        f"Return JSON array of strings.\n\n"
        f"Question: {question}\nJSON:")
    raw = resp.text.strip()
    if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
    try: return json.loads(raw)
    except: return [question]

print("Question Decomposition:\n")
for q in ["Is the Netsetos GenAI course worth the investment for a working professional?",
          "What are the current trends in AI agent development?"]:
    subs = decompose(q)
    print(f"  Q: {q}")
    for i, s in enumerate(subs): print(f"    {i+1}. {s}")
    print()


> **What just happened?** “Is the GenAI course worth it?” decomposed into: (1) What does the curriculum cover? (2) How does the price compare to alternatives? (3) What career outcomes do graduates report? (4) What are the prerequisites and time commitment? Each sub-question targets a different dimension. Searching for each separately yields better results than one broad query.


### Step 2 · Multi-Source Search + Source Evaluation

Search each sub-question, then score sources for credibility and relevance.

**`02_search_evaluate.py`** · _Block 2 of 3_

MULTI-SOURCE SEARCH + EVALUATION


In [ ]:
# MULTI-SOURCE SEARCH + EVALUATION
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# Simulated search results (in production: use Google Search API)
def search(query):
    db = {
        "curriculum": [{"source":"netsetos.com","text":"14 modules, 146hrs: transformers, RAG, agents, fine-tuning, MCP, GCP deployment.","type":"official"},
                       {"source":"reddit.com","text":"Took the course. Covers practical stuff, not just theory. Good projects.","type":"review"}],
        "price": [{"source":"netsetos.com","text":"14999 INR. EMI available. 5000 GCP credits included.","type":"official"},
                  {"source":"coursera.org","text":"Similar GenAI specialization: $49/month, ~6 months = $294 (~24500 INR).","type":"competitor"}],
        "careers": [{"source":"linkedin.com","text":"GenAI roles avg 15-40 LPA in India. 300% demand growth 2024-25.","type":"data"}],
    }
    for key, results in db.items():
        if key in query.lower(): return results
    return [{"source":"web","text":"No specific results","type":"general"}]

def evaluate_source(source, query):
    """LLM scores source credibility and relevance 1-10."""
    resp = model.generate_content(
        f"Score this source for the research query.\n"
        f"Query: {query}\nSource: {source['source']}\nText: {source['text']}\n"
        f"Return JSON: {{\"credibility\":1-10, \"relevance\":1-10, \"bias\":\"none/low/medium/high\"}}")
    raw = resp.text.strip()
    if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
    try: return json.loads(raw)
    except: return {"credibility":5,"relevance":5,"bias":"unknown"}

print("Search + Evaluate:\n")
for q in ["curriculum coverage", "price comparison"]:
    results = search(q)
    print(f"  Query: {q}")
    for r in results:
        score = evaluate_source(r, q)
        print(f"    [{r['source']}] cred:{score['credibility']} rel:{score['relevance']} bias:{score['bias']}")
    print()


### Step 3 · Full Deep Research Agent — End to End

Decompose → Search → Evaluate → Extract → Cross-check → Synthesize.

**`03_deep_research.py`** · _Block 3 of 3_

DEEP RESEARCH AGENT — Complete pipeline


In [ ]:
# DEEP RESEARCH AGENT — Complete pipeline
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class DeepResearchAgent:
    """Decompose → Search → Evaluate → Extract → Cross-check → Synthesize."""
    def __init__(self, search_fn):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.search = search_fn

    def _decompose(self, question):
        resp = self.model.generate_content(
            f"Break into 4 sub-questions. Return JSON array.\nQ: {question}")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return [question]

    def _extract_findings(self, sources):
        all_text = "\n".join(f"[{s['source']}]: {s['text']}" for s in sources)
        resp = self.model.generate_content(
            f"Extract key findings from these sources.\n"
            f"Return JSON: {{\"findings\":[\"...\"],\"contradictions\":[\"...\"],\"gaps\":[\"...\"]}}\n\n"
            f"Sources:\n{all_text}")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return {"findings":[all_text[:200]],"contradictions":[],"gaps":[]}

    def _synthesize(self, question, findings, sources):
        src_list = "\n".join(f"- {s['source']}: {s['text'][:80]}" for s in sources)
        resp = self.model.generate_content(
            f"Write a research report answering this question.\n\n"
            f"Question: {question}\n\n"
            f"Key findings: {json.dumps(findings['findings'])}\n"
            f"Contradictions: {json.dumps(findings.get('contradictions',[]))}\n"
            f"Gaps: {json.dumps(findings.get('gaps',[]))}\n"
            f"Sources:\n{src_list}\n\n"
            f"Write 3-4 paragraphs. Cite sources. Note contradictions. Identify gaps.")
        return resp.text.strip()

    def research(self, question):
        print(f"  1. Decomposing...")
        sub_qs = self._decompose(question)

        print(f"  2. Searching {len(sub_qs)} sub-questions...")
        all_sources = []
        for sq in sub_qs:
            results = self.search(sq)
            all_sources.extend(results)

        print(f"  3. Extracting from {len(all_sources)} sources...")
        findings = self._extract_findings(all_sources)

        print(f"  4. Synthesizing report...")
        report = self._synthesize(question, findings, all_sources)

        return {
            "report": report,
            "sub_questions": sub_qs,
            "sources": len(all_sources),
            "findings": findings,
        }

# ── Simulated search ──
def mock_search(q):
    db = [
        {"source":"netsetos.com","text":"14 modules, 146hrs, Python+GCP, 14999 INR, 85% completion, 4.8 stars."},
        {"source":"linkedin.com","text":"GenAI roles: 15-40 LPA India. 300% demand growth."},
        {"source":"coursera.org","text":"Similar courses: $294 (24500 INR) for 6-month specialization."},
        {"source":"reddit.com","text":"Practical projects, live classes help. Some say self-paced is enough."},
    ]
    return [s for s in db if any(w in q.lower() for w in ["course","price","career","worth","genai","curriculum","compare"])] or db[:2]

agent = DeepResearchAgent(mock_search)
print("Deep Research Agent:\n")
r = agent.research("Is the Netsetos GenAI course worth it for a working professional?")
print(f"\n  Sub-questions: {len(r['sub_questions'])}")
print(f"  Sources used: {r['sources']}")
print(f"  Findings: {len(r['findings']['findings'])}")
print(f"  Report:\n  {r['report'][:300]}...")


> **What just happened?** 4 sub-questions generated. Each searched across 4 sources. Findings extracted: curriculum depth, price comparison, career ROI. Contradictions noted: “some say self-paced is enough.” Final report synthesized with citations. This 6-step pipeline is the architecture behind Gemini Deep Research and Perplexity. The quality comes from decomposition + multi-source + cross-checking, not from a single clever prompt.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
